# Глава 8. Метрики
В данной главе рассмотрим основные метрики, используюшиеся в текстовой аналитике. Начнем с метрик перевода. Закончим метриками языковых моделей и поговорим про бенчмарки, их типизацию и рассмотрим примеры.

### Введение
Допустим мы обучили модель A. Но как мы понимаем, что модель A лучше модели B? У арифметической задачи `2+2=?` есть только один правильный ответ, остальные неверные (4.0, четыре, four). Проверить легко. Считаем долю правильных ответов. Квантили. 

Но как проверять генеративную модель? Здесь гораздо больше аспектов. Например, промпт "Напиши смешную историю", "Напиши HTTP-сервер".
в случае с генерацией критерий правильности агентными системами гораздо более многогранные

Вопрос приницпиальный не только для награждения победитиелей, это задает направление всему развитию области: модели оптимизируют то, что мы умеем измерять

В ML метрики:
- оптимизируемая
- отслеживать прогресс во время обучения: здесь нужна дешёвая, автоматическая, чувствительная метрика, которую можно считать на каждом шаге
- сравнивать готовые модели по их способностям: здесь важна не дешевизна, а то, насколько метрика отражает реальную полезность

Хочется чтобы метрика была
- универсальной
- устойчивой
- просто считаемой
- дифференцируемой
- выпуклой

### Закон Гудхарта
В 1975 году Чарльз Гудхарт, британский экономист, сформулировал принцип: `"Как только метрика становится целью, она перестаёт быть хорошей метрикой"`. Смысл в том, что любая метрика - это прокси к интересующей нас характеристике, на нее попмимо целевого факторов влияет множество второстепеныых и часто проще оптимизировать один из них, "хакнуть систему". Он сформулировал свой принцип относительно денежной политики, но принцип применим в любой сфере

Предположим, понимание студентом предмета $V$ оценивается оценкой за экзамен $U$. Тогда оценка $U = V + \varepsilon$ складывается из понимания + случайный шум: везение с билетом, натаскивание, списывание. Но дело в том, что вторую часть оптимизировать проще. Когда все понимают, как оценка зависит от $\varepsilon$, он перестаёт быть случайным. Его наращивают нарочно, например, натаскивают себя под формат теста, пишут шпаргалки и т.д. Важно, что понимание от этого никак не улучшается

Пример самого Гудхарта: центробанк отслеживал агрегат M3 как индикатор инфляции. Как только его сделали таргетом, финансовые институты перестроили инструменты так, что деньги перетекли в формы, не попадающие в M3, — индикатор перестал показывать то, что показывал. Есть таргетировать выпуск гвоздей в штуках, их выпускают мелкими, если в тоннах — выпускают мало, но огромных. Пара других примеров. Индекс Хирша и число публикаций как мера научной продуктивности породили дробление результата на минимальные публикуемые единицы, взаимное цитирование. Кол-во написанных строк кода как мера качества работы программиста. Число закрытых тикетов стимулирует дробление задач

Что можно сделать чтобы избежать "накрутки":
- использовать несколько метрик сразу, чтобы их было труднее взломать одновременно; 
- периодически менять метрики; 
- держать часть оценки закрытой (held-out), недоступной для оптимизации; 
- сочетать количественные показатели с качественным суждением; 
- различать метрики для мониторинга и метрики для управления — первые ломаются реже, потому что на них никто не давит

### Энтропия 
Рассмотрим языковую модель.
Не стоит путать с кросс-энтропией. Считаем метрику, но считаем но относительно себя а относительно конкретного использованного токена. 

## Перпелския

Языковая модель прогнозирует вероятность следующего токена. Естественная мера качества такой модели — насколько высокую вероятность она присваивает реальному тексту, которого раньше не видела. Если модель хорошо понимает язык, настоящие тексты должны быть для неё «ожидаемыми».

Формально это выражается через кросс-энтропию — среднее отрицательное лог-правдоподобие токенов на отложенной выборке:

$$H = -(1/N) \sum \log P(x_i | x_1, ..., x_{i-1})$$

Не стоит путать кросс-энтропию с энтропией генерации. Насколько модель может воспроизводить реальные тексты, чем больше, тем лучше. Энтропия генерации - сколько вариантов продолжения есть у модели на каждом шаге. В большинстве моделей регулируется температурой $\tau$ генерации

Перплексия — это просто экспонента от кросс-энтропии:

$$
PPL = e^{H}
$$

Мы рассматривали перплексию в первой главе

Интуиция следующая: это «эффективное число равновероятных вариантов», между которыми модель в среднем колеблется на каждом шаге. Перплексия 1 означала бы идеальное предсказание (модель всегда уверена в правильном токене), а перплексия, равная размеру словаря, — полное незнание языка (модель угадывает наугад). Чем ниже перплексия, тем лучше. Историческими ориентирами служили, например, наборы вроде Penn Treebank и WikiText, на которых десятилетиями мерили прогресс языкового моделирования.

У перплексии есть важная техническая оговорка. Она зависит от токенизации и словаря, поэтому перплексии двух моделей с разными токенизаторами напрямую несравнимы. Чтобы обойти это, используют метрики, нормированные на символы или байты — bits-per-character и bits-per-byte, — которые не зависят от того, как именно текст разбит на токены, и позволяют честно сравнивать разные архитектуры.

Сильные стороны перплексии — дешевизна и отсутствие необходимости в разметке: её можно считать на любом корпусе и строить по ней кривые обучения. Именно поэтому она остаётся главной метрикой на этапе предобучения. Слабость в том, что перплексия измеряет правдоподобие и беглость, но не полезность. Модель может прекрасно предсказывать токены и при этом плохо следовать инструкциям, врать или быть бесполезной в диалоге. Поэтому с переходом к инструктивным и диалоговым моделям перплексия перестала быть достаточной, и центр тяжести сместился к внешним, поведенческим метрикам.

## Метрики перевода
Задача машинного перевода была одной из первых практических задач в NLP. Первые идеи были сформулированы еще в 1949 году. В 1954 году в ходе Джорджтаунского эксперимента IBM перевели 49 русских предложений на английский язык, использовался словарь в 250 слов и 6 правилами. Пракическая демонстрация возможностей вызвала волну интереса и финансирования. Однако к 1966 году машинный перевод признали медленнее и дороже человеческого и область затормозилась почти на 20 лет. Статистического модели языка развитие возобновилось. Эксперты начали называть задачу решенной толькок к концу 2010-х.

У МП есть несколько похожих задач, можно рассматривать как seq2seq генерацию. Близкая задача - это задача аннотирования / суммаризации. Входной и выходной языки здесь не отличаются, нужно длинную последовательность перевести в короткую с максимальным сохранением смысла 

<img src="img/metrics.png" width=500>

До появления автоматизированных метрик перевод преимущественно оценивалася экспертно (по шкале 1-5). С ростом кол-ва моделей процесс оценивания сделали автоматизированым. Главное, чтобы был эталонный ответ (референс), размеченный профессиональным переводчиком

Общая задача генерации формально подходит, но сложно предложить эталонный ответ.

Какие сложности: перевод почти никогда не отображет слова 1-к-1. Порядок слов может быть разный.

На чем может быть основана метрика:
- пересечение
- сопоставление
- расстояние
    - в смысле кол-ва исправлений, редакторское
    - непрервыное
- семантика
    - табличная (WordNet)
    - непрерывная (эмбединги)

Исторически компания IBM довольно много занималась переводом 

Начнем с самых простых метрик, основанных на подсчете пересечения перевода с эталоном. Они опираются на точность (precision) и полноту (recall) перевода:

$$P=\frac{|T| \cap |R|}{|T|} \quad R = \frac{|T| \cap |R|}{|R|} \text{, где T - множество токенов перевода, R - множество токенов эталона}$$ 

### BLEU
В 2001 году исследователи из IBM ([Papineni et al.](https://aclanthology.org/P02-1040/)) предложили метрику **BLEU** для оценки качества машинного перевода. Идея простая — сравниваем результат модели с эталонным переводом, предоставленным профессиональным переводчиком

Считалось, что в переводе важнее точность, так как у перевода много допустимых вариантов. Поэтому BLEU ориентирована на точность.

Кроме того она изначально была задумана как "корпусная", оценивается система целиком, а не отдельное предложение. Поэтому мы суммируем по всем предложениям корпуса $\sum_{C \in \text{Candidates}}$, причём до деления: сначала складываем все совпадения по корпусу, потом все n-граммы, и только потом делим. Это не то же самое, что усреднить $p_n$ по предложениям: короткие предложения не получают завышенного веса, а редкие нули по 4-граммам растворяются в общей сумме

$$p = \frac{\sum |T| \cap |R|}{\sum |T|}$$

Модель может жульничать с повторами. Например, перевод «кот кот кот кот» при эталоне «кот сидит на коврике» дает точность 100%, ведь все 4 униграммы встречаются в эталоне, $p_1 = 1$. По этой причине вклад повторяющихся токенов ограничивают сверху числом его вхождений в эталон (clipping).

В переводе особую роль играют выражения и словосочетания, поэтому сравнивают не только отдельные слова, но и n-граммы. Таким образом, получаем четыре показателя: $p_1$, $p_2$, $p_3$ и $p_4$. Почему: униграммы проверяют, что модель взяла правильные слова, а n-граммы, что она поставила их в правильном порядке. 

Получив четыре точности $p_1..p_4$, их надо свернуть в одно значение. Авторы берут геометрическое среднее вместо арифметического:
$$\prod_{n=1}^{N} p_n^{w_n}$$

Почему: геометрическое гораздо более чувствительно. Если арифметическое компенсирует провал на одном уровне успехом на другом, то геометрическое требует быть хорошим одновременно на всех уровнях - порядок слов важен. Для сравнения, арифметическое среднее дает $\sum {0.8 + 0.6 + 0.05 + 0.01} = 0.365$, геометрическое: $\sqrt[4]{0.8 \cdot 0.6 \cdot 0.05 \cdot 0.01} = 0.083$

На практике это произведение переписывают через логарифм, получаем то же самое число, но меньше проблем при перемножении маленьких дробей:
$$\prod_{n=1}^{N} p_n^{w_n} = \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

где $p_n$ — точность по n-граммам длины $n$, $w_n$ — веса (обычно $w_n = 1/N$, то есть $1/4$)

При этом выражение перестает считаться при $p_n = 0$, так как не определен логарифм $\log 0 = -\infty$, но это крайне редкая ситуация (когда не совпало ничего), и чтобы обезопастить себя, можно использовать сглаженный вариант формулы smoothed BLEU (Chen & Cherry), где добавляют малую константу к счётчикам

Еще один момент, который нужно учесть, метрика не проверяет длину перевода. Можно выдать одно слово, в котором модель уверена, и получить $p_n = 1$. Поэтому к ней добавляют штрафной мультипликатор BP:

$$\text{BLEU} = BP \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

где $c$ — суммарная длина кандидатов, $r$ — суммарная длина референсов:

$$BP = \begin{cases} 1 & \text{если } c > r \\ e^{(1 - r/c)} & \text{если } c \le r \end{cases}$$

За слишком длинные переводы отдельного штрафа нет, но в этом случае лишние n-граммы и так попадают в знаменатель $p_n$, чем снижают точность

Значение метрики зависит от токенизации, регистра, обработки пунктуации. SacreBLEU - стандартизованная реализация метрики BLEU 

Есть вариант метрики от Google, называется __GLEU__, считается как минимум из точности и полноты по n-граммам. Пусть $H$ — мультимножество всех n-грамм гипотезы для $n = 1..N$ (обычно $N=4$), $R$ — то же для референса
$$\text{GLEU} = \min \bigg( \frac{|H \cap R|}{|H|}, \frac{|H \cap R|}{|R|} \bigg)$$

### LEPOR 
композиция из штрафа за длину, штрафа за расхождение позиций слов и гармонического среднего precision/recall. 

hLEPOR — версия с настраиваемыми весами компонентов. Идея — собрать в одной формуле то, что BLEU игнорирует (recall, позиции) и METEOR учитывает лишь частично.

### ROUGE
В отличие от перевода задача суммаризации чаще оценивается не по точности, а по полноте (recall). Действительно, не так критично, если в суммаризацию попадут какие-то лишние слова, которых нет в эталоне, куда важнее, чтобы сумаризация содержала необходимый минимум ключевых слов из оргинала. 

В 2004 году (Chin-Yew Lin et al) взяли логику метрики BLEU и заточили ее под задачу суммаризации. Так возникло семейство метрик **ROUGE** (Recall-Oriented Understudy for Gisting Evaluation). Тот факт что эта метрика зеркальна метрике BLEU получл отражение в названии. Ключевое отличие в том, что теперь мы считаем совпадение относительно эталона, а не относительно кандидата:

$$\text{ROUGE(N)} = \frac{\sum_{S \in \{Refs\}}\sum_{\text{ngram} \in S} Count_{match}(ngram)}{\sum_{S \in \{Refs\}}\sum_{\text{ngram} \in S} Count(ngram)}$$

Обратная уязвимость тоже есть: Recall не наказывает за длину, и система, скопировавшая весь документ, получит Recall = 1. Поэтому на практике репортят F-меру:

$$F_\beta = \frac{(1+\beta^2) \cdot P \cdot R}{\beta^2 \cdot P + R}$$

При $\beta = 1$ — обычная F1. Это и есть та роль, которую в BLEU играл brevity penalty: ограничение длины. Только BLEU делает это отдельным множителем, а ROUGE — вторым членом F-меры

В суммаризации нет требования дословного совпадения фраз, как в переводе, поэтому ограничиваются расчетом ROUGE-1 и ROUGE-2

__ROUGE-L:__ длиннейшая общая подпоследовательность

Проблема n-грамм: они требуют непрерывного совпадения. Референс «кот сидел на коврике», кандидат «кот тихо сидел на коврике» — биграмма «кот сидел» разрушена вставкой, хотя порядок слов сохранён идеально.

ROUGE-L вместо n-грамм использует LCS (Longest Common Subsequence) — самую длинную последовательность слов, идущих в обоих текстах в одном порядке, но не обязательно подряд.

$$R_{lcs} = \frac{LCS(X, Y)}{|X|}, \qquad P_{lcs} = \frac{LCS(X, Y)}{|Y|}$$

$$F_{lcs} = \frac{(1+\beta^2) \cdot R_{lcs} \cdot P_{lcs}}{R_{lcs} + \beta^2 P_{lcs}}$$

где $X$ — референс длины $|X|$, $Y$ — кандидат длины $|Y|$.

В примере выше LCS = «кот сидел на коврике» = 4, и метрика видит полное покрытие. Плюс не нужно выбирать $n$ — метрика сама находит максимальную длину совпадения.

**ROUGE-Lsum** — вариант для многопредложенческих рефератов: LCS считается отдельно по каждому предложению референса и суммируется, чтобы порядок предложений не влиял на результат.

__ROUGE-S: skip-биграммы__

Компромисс между жёсткостью n-грамм и вседозволенностью LCS. Считаются пары слов в правильном порядке с любым разрывом между ними (или разрывом до $k$ — тогда ROUGE-S$k$).

«кот сидел на коврике» даёт skip-биграммы: (кот, сидел), (кот, на), (кот, коврике), (сидел, на), (сидел, коврике), (на, коврике) — 6 пар.

**ROUGE-SU** дополнительно добавляет униграммы, чтобы кандидат с одним верным словом не получал нулевой оценки.

Если есть несколько референсов, то берётся максимум ROUGE по всем эталонам

### METEOR
За простоту метрики BLEU приходится платить: она не учитывает возможную синонимчность слов, игнорирует полноту ответа и вообще плохо коррелирует с экспертной оценкой. В 2005 году [(Banerjee и Lavie)](https://aclanthology.org/W05-0909/) предложили усовершенствовать и назвали свою версию **METEOR** (Metric for Evaluation of Translation with Explicit ORdering). 

Ключевое нововведение в том, что вместо подсчёта совпадений строится карта соотвествия (alignment) между словами перевода и эталона. При подсчете кол-ва совпадений мы ищем сначала полные совпадения, потом ослабляем требование и смотрим основы слов (*бежал* ↔ *бежит*), синонимы по WordNet (*машина* ↔ *автомобиль*) и перефразировки по таблице парафраз. Если находится несколько соотвествий, выбираем ближайшее (minimum crossings), таким образом поощряются переводы, максимально сохраняюшие порядок.

Эксперименты показали, что экспертные оценки лучше коррелируют именно с полнотой, не с точностью. поэтому в качестве итоговой метрики взяли гармоническое среднее с сильным перекосом в сторону полноты (например, $\alpha = 0.9$):

$$\text{METEOR} = \frac{P \cdot R}{\alpha P + (1-\alpha) R}$$

Отдельные штрафы за длину не нужны, компонент Recall сам штрафует за короткий перевод, Precision за длинный.

METEOR также поощряет сохранение структуры. Однако делает это более гибко, через **чанки** (chunk) — максимальной группы слов, идущих подряд и в одном порядке в обоих текстах. Если перевод идеален, все слова образуют один чанк: $ch = 1$. Если слова верны, но порядок случаен, каждое слово — свой чанк: $ch = m$. Отношение $ch/m$ — степень фрагментации: чем ближе к 1, тем более рваный перевод. За фрагментацию надо платить штраф:

$$\text{METEOR} = \frac{P \cdot R}{\alpha P + (1-\alpha)R} \cdot \left(1 - \gamma \left(\frac{ch}{m}\right)^{\beta}\right)$$

Стандартно $\gamma = 0.5$, $\beta = 3$.  Множитель $\gamma$ ограничивает штраф сверху 50% — даже полностью перемешанный перевод теряет не больше половины оценки

Параметры $\alpha, \beta, \gamma$ подбирают под конкретную языковую пару, максимизируя корреляцию с человеческими оценками на данных WMT. В более поздних версиях метрики добавили ещё и веса по типам совпадений (например $W_{exact}$ важнее, чем $W_{paraphrase}$ и т.д.) и дискаунтинг функциональных слов.

Метрика __RIBES__ (Rank-based Intuitive Bilingual Evaluation Score) так же основана на сопоставлении слов (alignment map), но она призвана не ограничиваться локальным порядком, а глобальным. Поэтому измеряют не пересечением, а таким показателем как ранговая корреляция Кендалла. Создана для языковых пар с сильно различающимся порядком слов (японский↔английский), где BLEU не чувствителен к перестановкам на дальних дистанциях.

[(Popovic, 2015)](https://aclanthology.org/W15-3049.pdf) предложила считать совпадения не слов, а символьных n-граммов, такую метрику назвали __chrF__.

$$\text{chrF}\beta = (1 + \beta^2) \cdot \frac{\text{chrP} \cdot \text{chrR}}{\beta^2 \cdot \text{chrP} + \text{chrR}}$$

Следующая группа метрик основана на перестановках слов

### Edit Distance
Метрика __WER__ (Word Error Rate) основана на вычислении расстония между переводами T и эталоном R. Расстояние - какое минимальное количество элментарных модификаций нужно сделать, чтобы привести перевод к эталону:
$$\mathrm{WER} = \frac{S + D + I}{N} \quad TER = \frac{(S + D + I + \text{Shifts})}{\text{N}}$$
где $S$ — кол-во замен слова (substitutions), $D$ - кол-во удалений слов, $I$ - кол-во вставок слова, $C$ — верные слова, $N$ - длина референса

Например, $\text{WER}(\text{"sdfs"}, \text{"sdfsdf"}) = 5$

Метрика - это нормированное расстояние Левенштейна на уровне слов

Метрика __TER__ (Translation Edit Rate) - то же самое, но разрешается еще одна модификация сдвиг целого блока слов:
$$TER = \frac{(S + D + I + \text{Shifts})}{\text{N}}$$

### NIST
Годом позже появилась модификация метрики BLEU, в которой к единичный вес совпдения заменили на более информативную обратную частоту (IDF). При агрегации показателей геометрическое среднее $\sqrt{p_1 p_2 p_3 p_4}$ заменили арифметическим $p_1 p_2 p_3 p_4$. Кроме того, штраф за краткость сделали более мягким

$$\sum_{n=1}^{N}
\left\{
\frac{\displaystyle\sum_{w_1\ldots w_n \in \text{match}} \mathrm{Info}(w_1\ldots w_n)}
{\displaystyle\sum_{w_1\ldots w_n \in \text{hyp}} 1}
\right\}
\cdot \exp\!\left\{\beta \log^2
\left[\min\!\left(\frac{L_{\mathrm{hyp}}}{\bar{L}_{\mathrm{ref}}},\, 1\right)\right]\right\}$$

$$\text{где } \mathrm{Info}(w_1 \ldots w_n) =
\log_2 \frac{\mathrm{count}(w_1 \ldots w_{n-1})}{\mathrm{count}(w_1 \ldots w_n)}$$

Метрика NIST действительно коррелировала лучше с человеческими оценками, но заменить BLEU так и не смогла. Во-первых, она все-таки требовала расчета обратных частот по всему корпусу. Во-вторых шкала не интерпретируема.

### LEPOR
ToDO

## Нейросетевые метрики
С начала 2000-х модели ушли далеко вперед в плане учета семантики, а метрики все оставались синтаксическими (сравнивали на уровне строк). А что если сравнивать в семантическом пространстве?

### BERTScore 
В 2019 году один из первых вариантов такой метрики предложили [(Zhang et al., 2019)](https://arxiv.org/abs/1904.09675), которые задействовали для этого популярную на тот момент артхитектуру модели BERT, и назвали метрику __BERTScore__

Модель сопоставляет перевод с эталоном не по самим токенам, а по их контекстным эмбеддингам - обогащенным описаниям, генерируемых как выход на последнем слое Трансформера непосредственно перед генерацией конкретного токена.

Почему: выходные эмбединги содержат всю богатую семантику токена, благодаря этому улавливает синонимию и перефразирование

Обозначим машинный перевод за $\hat{x} = \langle \hat{x}_1, ..., \hat{x}_k \rangle$, эталонный перевод за $x = \langle x_1, ..., x_m \rangle$$

Для каждого токена перевода считается расстояние до каждого токена эталона и выбирается наибольшее. Не всегда полное совпадение, важно еще чтобы контекст был одинаковым.
Вместо точного совпадения токенов — мягкое соответствие. Для каждого токена берём самый похожий токен в другом тексте:

$$R_{BERT} = \frac{1}{|x|}\sum_{x_i \in x} \max_{\hat{x}_j \in \hat{x}} x_i^\top \hat{x}_j$$

$$P_{BERT} = \frac{1}{|\hat{x}|}\sum_{\hat{x}_j \in \hat{x}} \max_{x_i \in x} x_i^\top \hat{x}_j$$

$$F_{BERT} = 2\,\frac{P_{BERT} \cdot R_{BERT}}{P_{BERT} + R_{BERT}}$$

Векторы предварительно нормированы, поэтому скалярное произведение — это косинус. Структура формул та же, что в ROUGE-1: Recall — покрытие референса, Precision — оправданность кандидата. Отличие только в том, что вместо индикатора «совпало/не совпало» стоит вещественное число от 0 до 1.

Сопоставление жадное, а не оптимальное — каждый токен независимо выбирает лучшую пару. Это дешевле полноценного венгерского алгоритма и на практике работает не хуже

Опционально токены можно взвесить по обратной частоте: совпадение по слову *«и»* будет весить, чем по слову *«ратификация»*. Так смешиваются семантика и классический полход на частотности

$$R_{BERT} = \frac{\sum_{x_i \in x} idf(x_i)\max_j x_i^\top \hat{x}_j}{\sum_{x_i \in x} idf(x_i)}$$

Сырые косинусы лежат в узком диапазоне — случайные предложения дают около 0.7–0.8, и различия между системами выглядят микроскопическими. Поэтому для лучшей интерпретируемости оценку линейно растягивают относительно базового уровня $b$, посчитанного на случайных парах:
$$\hat{F} = \frac{F_{BERT} - b}{1 - b}$$



### BLEURT

Несмотря на прорыв относительно базовых метрик, метрика BERTscore изначально не обучалась на оценку качества. Она просто использует готовую языковую модель как источник представлений. Но семантическая близость не всегда равно качество перевода: модель может не знать, например, что пропуск отрицания катастрофичен, а перестановка придаточного — нет

[(Sellam et al., 2020)](sdsd) предложили, давайте  моделировать человеческую оценку напрямую. Изобретать велосипед не нужно, у нас есть трансформерная модель для работы стекстами. Так появилась метрика BLEURT

Берём BERT, подаём на вход сконкатенировать пару  (кандидат, референс) в один текст, а поверх `[CLS]`-токена вешаем один линейный слой:

$$\text{BLEURT} = W\tilde{v}_{[CLS]} + b$$

Данную модель обучаем под задачу регрессии на человеческих оценках с МНК функцией потерь

Основная проблема - человеческих оценок мало. Их десятки тысяч, они дорогие, привязаны к конкретным годам и языковым парам. Модель, обученная только на них, переобучается и не переносится на новые системы

Ключевая идея BLEURT. Между обычным BERT-предобучением и файнтюном на людях вставляется этап на миллионах синтетических пар.

Пары генерируются из предложений Википедии:
- маскирование и восстановление через BERT
- обратный перевод (back-translation)
- случайное удаление слов

Каждая пара размечается автоматически — набором дешёвых сигналов сразу:
- BLEU, ROUGE, BERTScore между парой
- вероятности обратного перевода
- entailment-метки от модели NLI (следует / противоречит / нейтрально)

Модель обучается предсказывать все эти сигналы одновременно (multi-task). Смысл: она заранее выучивает, что вообще бывает с текстом и какие искажения важны, а потом дообучается на людях уже подготовленной

### COMET

[(Rei et al., 2020)](https://arxiv.org/abs/2009.09025) продолжили ту же логику, но к кандидату и референсу добавили еще оригинал ппредложение. Метрику назвали COMET

Мотивация следующая: вариантов перевода много, не обязательно привзываться к рефернсу. Если модель перевела иначе, но верно относительно оригинала, референс этого не покажет, а источник покажет. Это позволяет снять фундаментальное ограничение предыдущих метрик, включая BLEURT

<img src="img/comet.png" width=350>

Все три сегмента кодируются раздельно одним многоязычным энкодером (XLM-R), затем пулингом сворачиваются в векторы $s, h, r$. Из них собирается вектор признаков:

$$x = [h;\ r;\ h \odot s;\ h \odot r;\ |h - s|;\ |h - r|]$$

Здесь $\odot$ — поэлементное произведение, $|\cdot|$ — модуль разности. Произведение улавливает согласованность, разность — расхождение. Дальше feed-forward сеть выдаёт скаляр, обучение — регрессия на человеческих оценках (DA, MQM).

Раздельное кодирование — важное отличие от BLEURT: сегменты не видят друг друга внутри энкодера, взаимодействие вынесено в явные признаки. Это делает эмбеддинги переиспользуемыми и ускоряет вычисление

Существует также альтернативная постановка: вместо регрессии обучаем пвнжирование на триплетах — перевод VS «лучший перевод», «худший перевод», и вытягиваем их в пространстве так, чтобы лучший был ближе к источнику и референсу. Функция потерь: Triplet margin loss. Даёт более робастное ранжирование, но менее интерпретируемые абсолютные значения

Если эталонного перевода вообще нет, можно убрать $r$ из входа, такая модификация называется **COMET-QE** (quality estimation), только по паре «оригинал — перевод».

### Общие ограничения класса

Непрозрачность: если в метрике BLEU сразу поянтно, откуда взялось число, COMET гораздо менее интерпретируема. Частично решает модификация xCOMET, которая заточена под интервальную разметку

Несопоставимость между версиями <br> Значение зависит от чекпойнта: COMET 0.85 на wmt20 и на wmt22 — разные вещи. Версию модели надо репортить всегда, как SacreBLEU репортит подпись токенизации.

Предвзятость<br>Метрика впитывает предпочтения переводчиков и систем, на которых обучалась

Сложность<br>если BLEU считается за миллисекунды на CPU, COMET-XXL GPU и минуты на тестсет

Что использовать<br> Рекомендация WMT последних лет: репортить нейросетевую метрику как основную (COMET или BLEURT) и chrF или BLEU как вспомогательную для сопоставимости с прошлыми работами. BLEU в одиночку на современных системах уже считается недостаточным.

Следующие метрики отдельные модели, обученные на человеческих оценках качества, то есть метрика буквально предсказывает, как ответ оценил бы человек

### LLM-as-a-judge
Наиболее универсальный вариант — подход __LLM-as-a-judge__, который к середине 2020-х стал доминирующим способом оценки открытой генерации. Берется какая-то мощная языковая модель (из топа), получает запрос, один или несоклько сгенерированных ответов для оценку + инструкцию по оцениванию, например, можно попросить выставить каждому решению свой score бал или просто выбрать лучший вариант.

У такого подхода к оценке появляется свойство масштабирумости - можно разметить миллионы генераций, не тратя много денег и времени. Качество при этом будет коррелировать с человеческой разметкой.

У модели оценщика однако тоже могут быть свои систематические искажения, о которых важно знать: 
- предпочтение собственного стиля и собственных ответов
- смещение в сторону более длинных и уверенно звучащих ответов
- чувствительность к порядку предъявления вариантов

Поэтому LLM-as_Judge важно калибровать. Как минимум перемешивать порядок вариантов и перепроверять часть примеров оценщиками

## Метрики классификации

Параллельно с метриками генерации всегда существовали и простые метрики для задач с проверяемым ответом, и именно они лежат в основе большинства современных бенчмарков. 

Для задач классификации это хорошо изветсные в машинном обучении метрики типа точности (__accuracy__), а также __precision__, __recall__ и __F-мера__

Для задач генерации кода ключевой метрикой стала __pass@k__: сгенерированный код запускают на модульных тестах и проверяют функциональную корректность, а pass@k оценивает вероятность того, что среди k попыток хотя бы одна пройдёт все тесты

Для задач математики (`2 x 2 = ?`) используют классификационные метрики - проверяется точное совпадение финального ответа. Если попали в точный ответ - задача решена, не попали - не решена

### Win Rate
Для сравнительных оценок ("какой из двух ответов лучше A или B?") — долю побед (__win rate__) одной модели над другой

Когда предполагается, что у задачи есть объективно проверяемый ответ, такие метрики наиболее надёжные. Сейчас вектор развития ИИ сместился в прикладную область: важно оценивать, не просто насоклько хорошо модель генерирует текст, а как она решает конкретные прикладные задачи (код, математика, агентные сценарии). В этих задачах, как правило, корректность можно проверить автоматически и однозначно

## Метрики читаемости
Иногда важно оценить "сложность" генерируемого текста. Для этого замеряют среднее кол-во слов в одном предложении, а также кол-во слогов или букв в одном слове. Для лучшей интерпретируемости значения иногда отображают в класс школы, которому соотвествует словарное разнообразие текста.

Либо можно посчитать долю слов с большим (3+) количеством слогов. Другой подход - оценить долю слов из топ-3000 наиболее популярных слов языка.

При этом важно учитывать, что устная речь отличается от письменной. Кроме того значение метрик сильно зависит от того, как транскрибатор нарежет устную речь на предложения

Так например, исследовали предвыборную речь Дональда Трампа и выяснили, что тот предпочитает использовать более короткие предложения по сравнению с другими кандидатами

## Метрики разнообразия
Нужны для оценки генерируемого текста. Доля уникальных n-грамм в числе всех сгенерированных. Тут правда следует быть аккуратным, чем длинне текст, тем больше знаменатель и тем быстрее падает метрика. На генерации какой длины показатель уникальности падает ниже порога
Доля неуникальных n-грамм позволяет отлавливать зацикливание

По генеративной модели:<br>
Энтропия (не путать с кросс-энтропией на обучающем тексте), сколько вариантов продолжения текста в каждый момент времени
Сюда же можно отнести коэффициент Ципфа распределения частот токенов
MAUVE сравнивает распределения эмбедингов естественного и сгенерированного текста KL дивергенцией

Новизна - доля n-грамм, не встречающихся в обучающей выборке или промпте. Оценивает генеративные обощающие способности модели
Отдельно можно выделить метрики суммаризации: доля слов из оригинала, средняя длина экстракта, степень суммаризации. Высокие значения coeerage density говорят об экстрактивном резюме, низкие что абстрактивная

## Технические метрики
Time to first token - время от начала обработки до начала генерации prefill
Time per output token
Inter-token latency
LAtency =FT + OT

Производительность
Throughtput - кол-во запросов в единицу времении, 
Goodput - кол-во успешных запросов в единицу вермени (уложившихся в time limit)
RPS

Вычисления
FLOP
MFU  сколько возможностей железа используется
HFU включая активаций
вычислений на прочитанный байт

Использщование памяти
веса + активации + KV-кэш
контекст
квантование

стоимость запроса или обратная метрика "токенов на 1$"
Большинство провайдеров кэширует, это сильно влияет на экономику

При выводе всех метрик слежует учитывать, что 
важны персентили а не среднее
нужно указывать сопутствующую нагрузку
замерять нужно на реальном траффике


## Бенчмарки

Бенчмарк — это стандартизованный набор данных и протокол оценки, позволяющий сравнивать разные модели между собой

Сначала была эпоха отдельных задач: каждый датасет (__SQuAD__ для вопросов-ответов, __SNLI__ для логического следования и так далее) мерил одну узкую способность. Затем пришла эпоха агрегации: бенчмарки __GLUE__ и его усложнённый наследник __SuperGLUE__ собрали россыпь задач на понимание языка в единый набор с одной сводной цифрой, чтобы измерять «понимание языка вообще». Характерно, что обе оценки были вскоре «решены» — модели превысили человеческий уровень, и это стало повторяющимся сюжетом.

Следующая эпоха — знания и рассуждения. Её определяющим бенчмарком стал __MMLU__: 57 предметов от школьного до профессионального уровня, от анатомии до юриспруденции. Рядом встали бенчмарки здравого смысла (__HellaSwag__, ARC, WinoGrande, PIQA) и правдивости (__TruthfulQA__, проверяющий, повторяет ли модель распространённые человеческие заблуждения). Отдельно развивались математика (__GSM8K__ — школьные задачи, MATH — олимпиадные) и код (HumanEval, MBPP)

Параллельно возникли мега-наборы и идея всесторонней оценки. __BIG-bench__ собрал более двухсот разнообразных задач, придуманных сообществом; из него выделили особо трудное подмножество BIG-bench Hard. Проект __HELM__ сместил акцент с одной цифры на многомерность: модель прогоняют по множеству сценариев и меряют не только точность, но и устойчивость, калибровку, смещения, эффективность. Это была важная смена философии — от «кто набрал больше» к «какова модель по совокупности свойств»

К середине 2020-х область столкнулась с кризисом насыщения. Классические бенчмарки — MMLU, HellaSwag, HumanEval — топовые модели стали проходить тесты с результатом выше 90%, и различия между ними утонули в шуме. Ответом стало новое поколение более трудных оценок:

- MMLU-Pro - усложнённый MMLU с десятью вариантами ответа вместо четырёх и обязательной цепочкой рассуждений (хотя к 2026 году и он подходит к насыщению)
- GPQA-Diamond - вопросы уровня PhD по биологии, физике и химии, специально составленные так, чтобы их нельзя было нагуглить; неспециалисты набирают около трети даже с доступом в интернет
- Humanity's Last Exam — около трёх тысяч вопросов экспертного уровня от специалистов разных областей, задуманные так, чтобы оставаться трудными несколько лет
- ARC-AGI и ARC-AGI-2 — задачи на абстракцию и обобщение, нацеленные на «текучий интеллект», а не на эрудицию; первая версия была фактически взята reasoning-моделями к концу 2024 года, что и потребовало второй
- Олимпиадная математика (__AIME__ и подобные) и проекты вроде FrontierMath для самого верхнего уровня

Отдельной и быстро растущей ветвью стали агентные бенчмарки, проверяющие не текст, а действие: SWE-bench и SWE-bench Verified (модель должна решить реальную задачу из репозитория на GitHub так, чтобы прошли тесты), а также GAIA, WebArena, AgentBench и tau-bench, оценивающие работу с инструментами, навигацию и многошаговые сценарии. По мере того как модели обретают агентность, оценка тоже трансформируется от "что модель говорит" к "что модель делает"

По измеряемой способности
- Знания и эрудиция;
- рассуждения;
- здравый смысл;
- математика;
- код;
- правдивость и безопасность;
- многоязычность;
- мультимодальность;
- работа с инструментами и агентность;
- длинный контекст

По протоколу предъявления<br>Здесь важен исторический сдвиг. В эпоху GLUE модель дообучали под каждую задачу и мерили дообученную версию. С появлением больших моделей перешли к оценке без дообучения, через формулировку запроса: zero-shot (без примеров), few-shot (несколько примеров прямо в контексте) и с явной просьбой рассуждать пошагово (chain-of-thought). Один и тот же бенчмарк может давать очень разные числа в зависимости от протокола, поэтому сравнивать модели корректно только в одинаковых условиях.

По способу выставления оценки
- автоматическая проверка по совпадению или тестам (дёшево и воспроизводимо, но применимо не везде)
- оценка моделью-судьёй (масштабируемо, но со своими искажениями)
- человеческая оценка (наиболее достоверна, но дорога и медленна)

По статичности. Классический бенчмарк — это фиксированный тестовый набор. Но фиксированный набор рано или поздно утекает в обучающие данные и теряет ценность. Поэтому возникли живые и состязательные форматы: непрерывно обновляемые лидерборды и подходы вроде Dynabench, где люди специально придумывают примеры, на которых текущие модели ошибаются.

По размерности результата. Одна сводная цифра (удобно для ранжирования, но скрывает компромиссы) против холистической многомерной оценки в духе HELM (точность, устойчивость, смещения, эффективность по отдельности).

Отдельно стоит выделить оценку по человеческим предпочтениям в формате арены, потому что это качественно иной подход. Вместо фиксированных задач с известными ответами он измеряет, какой ответ людям субъективно нравится больше. Самый известный пример — __LMArena__ (ранее известная как LMSYS Chatbot Arena): пользователю показывают ответы двух анонимных моделей на его собственный запрос, он выбирает лучший, а из миллионов таких попарных сравнений строится рейтинг по схеме [Elo](https://en.wikipedia.org/wiki/Elo_rating_system) или модели Брэдли–Терри. 

Сила арены в том, что она улавливает «ощущение полезности», которое не видят формальные бенчмарки. Слабость — в том же: люди также не лишины предвзятости: они склонны голосовать за более длинные, уверенные и красиво оформленные ответы, поэтому стиль может побеждать точность, и арена в этом случае измеряет "красоту", а не содержание

## Тест Needle in a Haystack
С развитием моделей произошел взрывной рост максимального окна контекста: 128 тысяч, 200 тысяч, а затем и миллионы токенов. Но возможность загружать длинный текст - это одно, а насколько эффективно модель использует этот контекст - другое

Идею теста "Needle in a Haystack" предложил Грег Камрадт в 2023 году: в длинный текст вставляют одно несвязанное с темой предложение (например, утверждение, что лучшее занятие в Сан-Франциско — съесть сэндвич в парке Долорес в солнечный день) и просят модель ответить на вопрос, связанный с этим утверждением (например, "Какое лучшее занятие в Сан-Франциско?"). Эксперимент многократно повторяют, меняя локацию и общую длину контекста. Затем статистику правильных ответов рисуют тепловой картой: по одной оси — длина, по другой — глубина, цвет ячейки показывает, правльно ли ответила модель

<img src="img/needle_in_a_haystack.png" width=300>

Обнаружили что:
- способность к извлечению деградирует с ростом длины (что довольно очевидно)
- почти всегда возникает эффект __Lost in the middle__: информацию в начале и в конце контекста модели находят почти безошибочно, а вот ближе к середине провал в качестве [(Liu et al, 2023)](https://arxiv.org/abs/2307.03172)

<img src="img/lost_in_the_middle.png" width=300>

Отчасти эффект может объясняться "осторожностью" модели. Так было, например, с [ранними тестами](https://www.anthropic.com/news/claude-2-1-prompting) Claude 2.1, который  набрал низкую точность из-за того, что был обучен не отвечать на основании информации, которую считает недостаточно достоверной. Переформулировка запроса заметно меняла результа

В дальнейшем тест эволюционировал в сторону усложнения. Появились варианты с несколькими фактами (__multi-needle__), требующие интегрировать разрозненную информацию. Возник бенчмарк __RULER__ с набором синтетических длинноконтекстных задач и увидели, что большинство моделей, проходивших оригниальный тест, валятся на нем. 

В том же направлении работают __NeedleBench__, __BABILong__ (рассуждение в длинном контексте), __NoLiMa__ (поиск без буквального совпадения слов) и LongBench

Существует множество методик, как бороьтся с неравномерностью внимания. Некоторые из них:
- __соритровка контекста__<br>расположение более релевантных документов ближе к началу/концу<br>повторное переранжирование отобранных документов более сложной полносвязной моделью
- __калибровка Attention__<br>добавляем числовую поправку, чтобы внимание больше веса давало токенам из середины контекста [(Hsieh et al, 2024)](https://arxiv.org/abs/2406.16008)<br>документы, которым модель часто дает больше внимания, перемещаем ближе к краю [(Peysakhovich et al, 2023)](https://arxiv.org/abs/2310.01427)<br>испольщование RoPE эмбедингов уменьшает деградацию на супер длинных контекстах [(Su et al, 2021)](https://arxiv.org/abs/2104.09864)
- __сжатие контекста__<br>выкидывание малозначимых токенов по перплексии, а-ля TF-IDF [(Jiang et al, 2023)](https://arxiv.org/abs/2310.05736)<br>замена кусков текста на их компактную суммаризацию
- __грамотное обучение__<br>расположение факта в разных локациях [(An et al, 2024)](https://arxiv.org/abs/2404.16811)<br>примеры с инструкциями на реально длинных контекстах
- __грамотный prompt engineering__<br>инструкцию дублируют в начале и конце<br>решают задачу по частям, потом агрегируют (в стиле map-reduce)
- __ориентир на точность__<br>на шаге Retrieval ставят выше порог релевантности - контекст получается меньше [(Lui et al, 2023)](https://arxiv.org/pdf/2407.01100)<br>информацию достают итеративно

## Проблемы и ограничения метрик

Наивная вера в цифры лидербордов — частая ошибка. Ниже список проблем, которые могут возникать у бенчмарков

__Утечки данных__<br>Data contamination - главная беда бенчмарков. Тестовые наборы публичны и со временем попадают в обучающие данные следующих моделей. Тогда высокий балл отражает не способность рассуждать, а запоминание ответов. Именно поэтому так ценятся свежие, приватные и состязательные оценки

__Насыщение__<br>У каждого бенчмарка ограниченный срок жизни: как только модели упираются в его потолок, он перестаёт различать сильнейших, и нужен новый, более трудный. Мы видели это на всей цепочке от GLUE до MMLU-Pro.

__Закон Гудхарта__<br>Когда бенчмарк становится целью оптимизации, под него начинают подгонять обучение, и высокий балл может расходиться с реальной полезностью. Модель учат «сдавать экзамен», а не быть умной

__Валидность__<br>Не всегда очевидно, что бенчмарк измеряет именно ту способность, которую заявляет; формат с выбором из вариантов, например, оставляет лазейки, которые модель может эксплуатировать, не понимая сути

__Разрыв с реальностью__<br>Высокий балл на академическом бенчмарке не гарантирует пользы в реальном продукте; корреляция между лидербордами и фактическим качеством работы бывает слабой, и нередко модели с более скромными общими баллами оказываются точнее на конкретных прикладных задачах.

__Воспроизводимость__<br>Результаты чувствительны к формулировке запроса, числу примеров, версии оценочного инструментария и способу нормализации ответа, поэтому числа из разных источников не всегда сравнимы напрямую.
